In [1]:
!pip install roboflow==1.1.48 --quiet
!pip install ultralytics --quiet
!pip install numpy==1.26.4 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is inc

In [4]:
!nvidia-smi

Sat Feb 14 15:31:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/content


In [3]:

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

# prevent ultralytics from tracking activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.0/112.6 GB disk)


In [5]:
from ultralytics import YOLO

from IPython.display import display, Image

In [6]:
pip install roboflow

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4
!pip install --force-reinstall opencv-python ultralytics


In [1]:
## custom

!pip install -q roboflow ultralytics opencv-python

from roboflow import Roboflow
from ultralytics import YOLO
import cv2
import json
from pathlib import Path

rf = Roboflow(api_key="SGwZ59PpBg8EbtkaZYTF")
project = rf.workspace("chatty").project("glove-detection-zhhg4")
dataset = project.version(1).download("yolov8")

data_yaml = f"{dataset.location}/data.yaml"

model = YOLO("yolov8n.pt")

model.train(
    data=data_yaml,
    epochs=20,
    imgsz=640,
    batch=16,
    name="glove_detection"
)

best_model_path = "runs/detect/glove_detection/weights/best.pt"

def run_inference(model_path, input_folder, output_folder="inference_output", conf=0.4):
    model = YOLO(model_path)
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    logs_path = output_path / "logs"

    output_path.mkdir(parents=True, exist_ok=True)
    logs_path.mkdir(parents=True, exist_ok=True)

    image_files = list(input_path.glob("*.jpg"))

    for img_path in image_files:
        image = cv2.imread(str(img_path))
        results = model.predict(str(img_path), conf=conf, verbose=False)[0]
        detections = []

        if results.boxes:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls_id = int(box.cls[0])
                label = model.names[cls_id]
                confidence = float(box.conf[0])

                color = (0,255,0) if "glove" in label else (0,0,255)

                cv2.rectangle(image,(x1,y1),(x2,y2),color,2)
                cv2.putText(image,f"{label} {confidence:.2f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,color,2)

                detections.append({
                    "label": label,
                    "confidence": round(confidence,4),
                    "bbox": [x1,y1,x2,y2]
                })

        cv2.imwrite(str(output_path / img_path.name), image)

        with open(logs_path / f"{img_path.stem}.json","w") as f:
            json.dump({"filename": img_path.name,
                       "detections": detections}, f, indent=4)

test_images_folder = f"{dataset.location}/test/images"

run_inference(best_model_path, test_images_folder)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to glove-detection-1 in yolov8:: 100%|██████████| 1353/1353 [00:00<00:00, 10321.48it/s]


100%|██████████| 6.25M/6.25M [00:00<00:00, 259MB/s]

New https://pypi.org/project/ultralytics/8.4.14 available 😃 Update with 'pip install -U ultralytics'


Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/glove-detection-1/data.yaml, epochs=20, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=glove_detection, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, 

...

      20/20      2.19G     0.6015     0.7153      1.195         16        640: 100%|██████████| 26/26 [00:07<00:00,  3.61it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<00:00,  3.48it/s]

                   all        160        179      0.886      0.917      0.949       0.75



20 epochs completed in 0.055 hours.
Optimizer stripped from runs/detect/glove_detection/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/glove_detection/weights/best.pt, 6.2MB

Validating runs/detect/glove_detection/weights/best.pt...
Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 168 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:03<00:00,  1.49it/s]


                   all        160        179      0.886      0.919      0.948      0.748
             bare_hand         74         74      0.926          1      0.995      0.841
            glove_hand         86        105      0.846      0.838      0.902      0.656
Speed: 0.4ms preprocess, 3.2ms inference, 0.0ms loss, 7.2ms postprocess per image
Results saved to runs/detect/glove_detection


In [2]:
from ultralytics import YOLO
import cv2
import json
from pathlib import Path

MODEL_PATH = "runs/detect/glove_detection/weights/best.pt"
INPUT_FOLDER = "test_images"
OUTPUT_FOLDER = "output"
CONFIDENCE = 0.4

model = YOLO(MODEL_PATH)

input_path = Path(INPUT_FOLDER)
output_path = Path(OUTPUT_FOLDER)
logs_path = output_path / "logs"

output_path.mkdir(parents=True, exist_ok=True)
logs_path.mkdir(parents=True, exist_ok=True)

image_files = list(input_path.glob("*.jpg"))

for img_path in image_files:

    image = cv2.imread(str(img_path))
    results = model.predict(str(img_path), conf=CONFIDENCE, verbose=False)[0]

    detections = []

    if results.boxes:
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            label = model.names[cls_id]
            confidence = float(box.conf[0])

            detections.append({
                "label": label,
                "confidence": round(confidence, 2),
                "bbox": [x1, y1, x2, y2]
            })

            color = (0, 255, 0) if "glove" in label else (0, 0, 255)
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(image, f"{label} {confidence:.2f}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.imwrite(str(output_path / img_path.name), image)

    json_data = {
        "filename": img_path.name,
        "detections": detections
    }

    with open(logs_path / f"{img_path.stem}.json", "w") as f:
        json.dump(json_data, f, indent=4)

print("Done")


Done


In [12]:
from ultralytics import YOLO
import cv2
import json
from pathlib import Path

MODEL_PATH = "runs/detect/glove_detection/weights/best.pt"
INPUT_FOLDER = f"{dataset.location}/test/images"
OUTPUT_FOLDER = "output"
CONFIDENCE = 0.4

model = YOLO(MODEL_PATH)

input_path = Path(INPUT_FOLDER)
output_path = Path(OUTPUT_FOLDER)
logs_path = output_path / "logs"

output_path.mkdir(parents=True, exist_ok=True)
logs_path.mkdir(parents=True, exist_ok=True)

image_files = list(input_path.glob("*.jpg"))[:3]

if not image_files:
    print("No .jpg images found in:", INPUT_FOLDER)
else:
    for img_path in image_files:

        image = cv2.imread(str(img_path))
        results = model.predict(str(img_path), conf=CONFIDENCE, verbose=False)[0]

        detections = []

        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls_id = int(box.cls[0])
                label = model.names[cls_id]
                confidence = float(box.conf[0])

                detections.append({
                    "label": label,
                    "confidence": round(confidence, 2),
                    "bbox": [x1, y1, x2, y2]
                })

                color = (0, 255, 0) if "glove" in label else (0, 0, 255)
                cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
                cv2.putText(image,
                            f"{label} {confidence:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,
                            color,
                            2)

        image_output_path = output_path / img_path.name
        cv2.imwrite(str(image_output_path), image)

        json_output_path = logs_path / f"{img_path.stem}.json"

        json_content = {
            "filename": img_path.name,
            "detections": detections
        }

        with open(json_output_path, "w") as f:
            json.dump(json_content, f, indent=2)

        print("Saved image:", image_output_path.resolve())
        print("Saved JSON :", json_output_path.resolve())

print("Done")


Saved image: /content/output/IMG20220226202121_jpg.rf.cc3e6a103cce7186f15a862fcb805185.jpg
Saved JSON : /content/output/logs/IMG20220226202121_jpg.rf.cc3e6a103cce7186f15a862fcb805185.json
Saved image: /content/output/1596468417313_jpg.rf.993467d9d622a74ba32c3beec2e38288.jpg
Saved JSON : /content/output/logs/1596468417313_jpg.rf.993467d9d622a74ba32c3beec2e38288.json
Saved image: /content/output/gloves-341_jpg.rf.4dbcb66fc60f29efdf3f6d07c69ab515.jpg
Saved JSON : /content/output/logs/gloves-341_jpg.rf.4dbcb66fc60f29efdf3f6d07c69ab515.json
Done
